# Data Cleaning Notebook
## Fraud Detection Project - Task 1a

This notebook handles data cleaning operations including:
- Loading raw datasets
- Handling missing values
- Removing duplicates
- Converting data types
- Saving cleaned data

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../src')

from data_loader import DataLoader
from data_cleaner import DataCleaner

# Set display options
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

%matplotlib inline

## 1. Load Raw Data

In [ ]:
# Initialize data loader
loader = DataLoader()

# Load datasets
fraud_df = loader.load_fraud_data('../data/raw/Fraud_Data.csv')
ip_mapping_df = loader.load_ip_mapping('../data/raw/IpAddress_to_Country.csv')
creditcard_df = loader.load_creditcard_data('../data/raw/creditcard.csv')

print("Data loaded successfully!")
loader.get_data_summary()

## 2. Inspect Fraud Data

In [ ]:
print("Fraud Data Shape:", fraud_df.shape)
fraud_df.head()

In [ ]:
# Data types
fraud_df.info()

In [ ]:
# Summary statistics
fraud_df.describe()

## 3. Missing Values Analysis

In [ ]:
# Check for missing values
missing_values = fraud_df.isnull().sum()
missing_pct = (missing_values / len(fraud_df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Percentage': missing_pct
})

missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

In [ ]:
# Visualize missing values
if missing_values.sum() > 0:
    plt.figure(figsize=(12, 6))
    sns.heatmap(fraud_df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
    plt.title('Missing Values Heatmap - Fraud Data')
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found in fraud data!")

## 4. Duplicate Detection

In [ ]:
# Check for exact duplicates
duplicate_rows = fraud_df.duplicated().sum()
print(f"Exact duplicate rows: {duplicate_rows}")

# Check for duplicate user_ids (could be legitimate)
duplicate_users = fraud_df['user_id'].duplicated().sum()
print(f"Duplicate user_ids: {duplicate_users}")

# Check for duplicate transactions (same user, same purchase time)
duplicate_transactions = fraud_df.duplicated(subset=['user_id', 'purchase_time']).sum()
print(f"Duplicate transactions (same user + time): {duplicate_transactions}")

## 5. Clean Fraud Data

In [ ]:
# Initialize cleaner
cleaner = DataCleaner()

# Handle missing values
fraud_df_clean = cleaner.handle_missing_values(fraud_df, strategy='auto')

# Remove duplicates
fraud_df_clean = cleaner.remove_duplicates(fraud_df_clean)

# Convert date columns
fraud_df_clean = cleaner.convert_data_types(
    fraud_df_clean, 
    datetime_cols=['signup_time', 'purchase_time']
)

# Validate age range (18-100)
fraud_df_clean = cleaner.validate_ranges(
    fraud_df_clean,
    {'age': (18, 100), 'purchase_value': (0, 10000)}
)

print("\n" + cleaner.get_cleaning_report())

In [ ]:
print(f"Original shape: {fraud_df.shape}")
print(f"Cleaned shape: {fraud_df_clean.shape}")
print(f"Rows removed: {fraud_df.shape[0] - fraud_df_clean.shape[0]}")

## 6. Clean Credit Card Data

In [ ]:
print("Credit Card Data Shape:", creditcard_df.shape)
creditcard_df.head()

In [ ]:
# Check for missing values in credit card data
cc_missing = creditcard_df.isnull().sum()
print("Missing values in credit card data:")
print(cc_missing[cc_missing > 0] if cc_missing.sum() > 0 else "No missing values")

In [ ]:
# Clean credit card data
cleaner_cc = DataCleaner()

creditcard_df_clean = cleaner_cc.handle_missing_values(creditcard_df, strategy='auto')
creditcard_df_clean = cleaner_cc.remove_duplicates(creditcard_df_clean)

print(f"\nOriginal shape: {creditcard_df.shape}")
print(f"Cleaned shape: {creditcard_df_clean.shape}")

## 7. Inspect IP Mapping Data

In [ ]:
print("IP Mapping Data Shape:", ip_mapping_df.shape)
ip_mapping_df.head()

In [ ]:
# Check for missing values in IP mapping
ip_missing = ip_mapping_df.isnull().sum()
print("Missing values in IP mapping data:")
print(ip_missing[ip_missing > 0] if ip_missing.sum() > 0 else "No missing values")

## 8. Save Cleaned Data

In [ ]:
# Save cleaned datasets
fraud_df_clean.to_csv('../data/processed/cleaned_fraud_data.csv', index=False)
creditcard_df_clean.to_csv('../data/processed/cleaned_creditcard.csv', index=False)

print("Cleaned data saved successfully!")
print(f"- fraud_df_clean: {fraud_df_clean.shape}")
print(f"- creditcard_df_clean: {creditcard_df_clean.shape}")

## 9. Data Quality Summary

In [ ]:
# Create quality report
quality_report = {
    'Dataset': ['Fraud Data', 'Credit Card Data'],
    'Original Rows': [fraud_df.shape[0], creditcard_df.shape[0]],
    'Cleaned Rows': [fraud_df_clean.shape[0], creditcard_df_clean.shape[0]],
    'Rows Removed': [
        fraud_df.shape[0] - fraud_df_clean.shape[0],
        creditcard_df.shape[0] - creditcard_df_clean.shape[0]
    ],
    'Removal %': [
        ((fraud_df.shape[0] - fraud_df_clean.shape[0]) / fraud_df.shape[0]) * 100,
        ((creditcard_df.shape[0] - creditcard_df_clean.shape[0]) / creditcard_df.shape[0]) * 100
    ]
}

quality_df = pd.DataFrame(quality_report)
quality_df

## Summary

### Data Cleaning Completed

**Fraud Data:**
- Handled missing values
- Removed duplicates
- Converted date columns to datetime format
- Validated age and purchase value ranges

**Credit Card Data:**
- Checked for and handled missing values
- Removed duplicates

**Next Steps:**
- Exploratory Data Analysis (EDA)
- Feature Engineering
- Class Imbalance Handling